### CatBoost

- CatBoost processes raw categorical text directly, completely eliminating the need for OneHotEncoder or TargetEncoder.

- No ColumnTransformer or preprocessing required, just pass the raw data and define text columns using cat_features.

- It uses Ordered Target Statistics (OTS), dynamically calculating category target averages sequentially to mathematically prevent data leakage.

- CatBoost natively routes numeric missing values (NaN) through its decision trees without crashing, so we skip imputation too.

- It automatically detects and merges related categorical features (day="Sunday" + color="Yellow") during tree splits to find complex patterns.

- It builds symmetric decision trees, which restricts structural complexity and makes the model incredibly stable on test data.

- Bundling features, labels and categories into a Pool object compiles the dataset into a highly optimized C++ memory block for fast training speeds.

---
## <u>Import functions and Load dataset

In [1]:
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from catboost import CatBoostClassifier, Pool

# 1. Load Titanic Dataset
data = sns.load_dataset("titanic")

# Label = "survived", redundant feature = "pclass", data leakage feature = "alive"
X = data.drop(columns=["survived", "alive", "pclass"])
y = data["survived"]

---
## <u>Handle categorical data</u>

In [2]:
# No need to use any encoders, just define the categorical features and clean missing values (nan) from data

cat_features = ['sex', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alone']

# First we convert the cat_features into string so that we can replace "nan" with "missing"

for col in cat_features:
    X[col] = X[col].astype(str).replace('nan', 'Missing')

---
## <u>Train Test Split</u>

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

---
## <u>Pooling</u>

In [6]:
# Pools Bundle the data, labels, and the list of categorical features into  single, highly optimized C++ memory block.
# This makes training exponentially faster.

train_pool = Pool(data=X_train, label=y_train, cat_features=cat_features)
test_pool = Pool(data=X_test, label=y_test, cat_features=cat_features)

---
## <u>Create model</u>

In [11]:
cbc_model = CatBoostClassifier(
    iterations=200,       # Number of sequential trees to build  
    learning_rate=0.01,   
    max_depth=4,              
    l2_leaf_reg=5,        # Heavy L2 regularization on leaf weights
    random_state=42,     
    verbose=0            
)

---
## <u>Train and Predict</u>

In [12]:
# Because we used Pools, we just pass the Pool object directly, No need to pass X and y separately
cbc_model.fit(train_pool)
y_test_pred = cbc_model.predict(test_pool)
y_train_pred = cbc_model.predict(train_pool)

---
## <u>Evaluate</u>

In [13]:
# check both Train and Test to monitor the Bias-Variance tradeoff (checking for overfitting).

print("for Catboost Classifier (baseline)")
print("\nFor training data :-")
print("Train Accuracy : ", accuracy_score(y_train, y_train_pred))
print("Classification Report :\n", classification_report(y_train, y_train_pred))
print("\nFor testing data :-")
print("Test Accuracy : ", accuracy_score(y_test, y_test_pred))
print("Classification Report :\n", classification_report(y_test, y_test_pred))

for Catboost Classifier (baseline)

For training data :-
Train Accuracy :  0.8314606741573034
Classification Report :
               precision    recall  f1-score   support

           0       0.84      0.90      0.87       444
           1       0.82      0.71      0.76       268

    accuracy                           0.83       712
   macro avg       0.83      0.81      0.82       712
weighted avg       0.83      0.83      0.83       712


For testing data :-
Test Accuracy :  0.8100558659217877
Classification Report :
               precision    recall  f1-score   support

           0       0.81      0.88      0.84       105
           1       0.80      0.72      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179

